<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/modal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modal Cookbook

In [ ]:
%pip install llama-index llama-index-llms-modal

[Modal](https://modal.com?utm_source=llamaindex&utm_medium=referral&utm_campaign=inference_partners&utm_content=home_page) serves open models behind OpenAI-compatible inference APIs:

1. [Model APIs](https://modal.com/docs/guide/model-apis?utm_source=llamaindex&utm_medium=referral&utm_campaign=inference_partners&utm_content=model_apis_guide) are pay-per-token endpoints on Modal-managed capacity, reached through a shared regional gateway (`https://inference.<region>.modal.direct/v1`). The model ID is the hostname shown in the dashboard, e.g. `my-endpoint.us-west.modal.direct`.

2. [Endpoints](https://modal.com/docs/guide/endpoints?utm_source=llamaindex&utm_medium=referral&utm_campaign=inference_partners&utm_content=endpoints_guide) are dedicated-capacity deployments with their own base URL, for autoscaling production workloads and custom weights.

#### Instantiation

Create a proxy token and grant it access to your Model API's or Endpoint's environment:

```bash
modal workspace proxy-tokens create
modal workspace proxy-tokens allow <token-id> <environment-name>
```

In [2]:
from llama_index.core.llms import ChatMessage
from llama_index.llms.modal import Modal

llm = Modal(
    model="my-endpoint.us-west.modal.direct",
    region="us-west",  # us-west, us-east, ca-central, eu-west, ap-south, ...
    proxy_token="wk-<id>.ws-<secret>",
    max_tokens=8192,
)

# or a dedicated Endpoint:
# llm = Modal(
#     model="Qwen/Qwen3.5-4B",
#     endpoint_url="https://my-endpoint.us-west.modal.direct",
#     proxy_token="wk-<id>.ws-<secret>",
# )


#### Call `complete` with a prompt

In [3]:
response = llm.complete("What is Modal?")
print(response.text)

"Modal" most commonly refers to **Modal Labs**, a serverless cloud platform for running code, especially popular for AI and machine learning workloads. You write Python functions, add a decorator, and Modal handles the infrastructure—spinning up containers, allocating GPUs, and scaling to zero when idle. It's often used for fine-tuning models, running inference endpoints, batch processing, and other compute-heavy jobs without managing servers yourself.

Depending on context, "modal" can also mean:

- **Modal fabric** – a soft, breathable semi-synthetic textile made from beech tree pulp (a type of rayon)
- **Modal (UI/UX)** – a dialog or window that blocks interaction with the rest of an app until it's dismissed
- **Modal logic** – a branch of logic dealing with possibility and necessity
- **Modal verbs** – words like *can*, *must*, and *should* in grammar
- **Modal music** – music based on modes like Dorian or Phrygian rather than major/minor scales

Happy to go deeper on any of these!

#### Call `chat` with a list of messages

In [4]:
messages = [
    ChatMessage(role="system", content="You are a concise assistant."),
    ChatMessage(role="user", content="Name three benefits of running LLM inference on serverless GPUs."),
]
response = llm.chat(messages)
print(response.message.content)

1. **Cost efficiency** – Pay-per-use billing (often per second) means you only pay for GPU time during actual inference, avoiding idle costs on always-on servers.

2. **Automatic scaling** – The platform scales from zero to many instances to handle traffic spikes, with no manual capacity planning or provisioning.

3. **No infrastructure management** – No need to maintain GPU hardware, drivers, or CUDA environments; you just deploy your model and code.


## Streaming

#### `stream_complete`

In [5]:
response = llm.stream_complete("Explain serverless GPUs in two sentences.")
for r in response:
    print(r.delta, end="")

Serverless GPUs are a cloud computing model that provides on-demand access to GPU hardware without requiring users to provision, configure, or maintain the

 underlying infrastructure. Resources automatically scale with workload demands and are billed only for actual usage—typically per second or per millisecond—making them cost-effective for intermittent

 or bursty AI/ML tasks like model inference and training.

#### `stream_chat`

In [6]:
messages = [
    ChatMessage(role="user", content="Write a haiku about autoscaling."),
]
response = llm.stream_chat(messages)
for r in response:
    print(r.delta, end="")

Traffic swells at dawn—
servers bloom to meet the load,
at night, they all sleep.